# Koo an's EDA

Roostoo universe (~67 tokens).

Loads **`data/research_roostoo_1m.duckdb`** and analyzes **all** symbols (cross-sectional).

**Database refresh** lives in **`research/db/`** — see `research/db/README.md` and run `load_1m_roostoo_universe.py` or open `inspect_duckdb.ipynb` there.

Run cells top-to-bottom. Cwd can be repo root or `research/`.

In [ ]:
# Install/import core packages used throughout the notebook (EDA + classification)
import sys
import subprocess

# NOTE: keep this at the very top so downstream cells don't need to re-import.
for pkg in (
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
    "duckdb",
    "requests",
):
    try:
        __import__(pkg)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import requests

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

# Define a simple on-disk cache for API responses to avoid rate limits
import hashlib
import json
import os
import time
from pathlib import Path
from urllib.parse import urlencode

API_CACHE_DIR = Path("outputs/api_cache")
API_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def _cache_key(url: str, params: dict | None = None, headers: dict | None = None) -> str:
    """Create a stable cache key from request components (excluding sensitive headers)."""
    params = params or {}
    # Never include API keys in cache key; strip common auth headers
    headers = headers or {}
    safe_headers = {
        k.lower(): v
        for k, v in headers.items()
        if k.lower() not in {"authorization", "x-cmc_pro_api_key", "api-key", "x-api-key"}
    }
    payload = {
        "url": url,
        "params": params,
        "safe_headers": safe_headers,
    }
    raw = json.dumps(payload, sort_keys=True, separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


def cached_get_json(
    url: str,
    params: dict | None = None,
    headers: dict | None = None,
    ttl_seconds: int | None = None,
    session: "requests.Session | None" = None,
    timeout: int = 60,
    cache_subdir: str = "default",
    force_refresh: bool = False,
):
    """HTTP GET returning JSON, cached to disk.

    - ttl_seconds=None means cache forever.
    - force_refresh=True bypasses cache.
    """
    import requests  # local import so this utility cell stands alone

    params = params or {}
    headers = headers or {}

    subdir = API_CACHE_DIR / cache_subdir
    subdir.mkdir(parents=True, exist_ok=True)

    key = _cache_key(url, params=params, headers=headers)
    cache_path = subdir / f"{key}.json"
    meta_path = subdir / f"{key}.meta.json"

    if (not force_refresh) and cache_path.exists() and meta_path.exists():
        try:
            meta = json.loads(meta_path.read_text(encoding="utf-8"))
            ts = meta.get("cached_at", 0)
            if ttl_seconds is None or (time.time() - ts) <= ttl_seconds:
                return json.loads(cache_path.read_text(encoding="utf-8")), {"cached": True, **meta}
        except Exception:
            # If cache is corrupt, fall through to refetch
            pass

    sess = session or requests.Session()
    resp = sess.get(url, params=params, headers=headers, timeout=timeout)

    # Cache only successful JSON responses
    try:
        data = resp.json()
    except Exception:
        resp.raise_for_status()
        raise

    meta = {
        "cached_at": time.time(),
        "url": url,
        "params": params,
        "status_code": resp.status_code,
    }

    if resp.status_code == 200:
        cache_path.write_text(json.dumps(data, ensure_ascii=False), encoding="utf-8")
        meta_path.write_text(json.dumps(meta, ensure_ascii=False), encoding="utf-8")
        return data, {"cached": False, **meta}

    # If not 200, don't overwrite cache; raise to caller
    resp.raise_for_status()


print(f"API cache dir: {API_CACHE_DIR.resolve()}")
print("Defined: cached_get_json(url, params, headers, ttl_seconds, cache_subdir, force_refresh)")


In [ ]:
def repo_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(10):
        if (p / "research" / "Koo an's EDA.ipynb").exists() and (p / "data").is_dir():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise RuntimeError(
        "Could not find repo root (expected research/Koo an's EDA.ipynb + data/). "
        f"cwd={Path.cwd().resolve()}"
    )


ROOT = repo_root()
DB_FILE = ROOT / "data" / "research_roostoo_1m.duckdb"

if not DB_FILE.exists():
    raise FileNotFoundError(
        f"Missing {DB_FILE}\n"
        "Build from repo root — see research/db/README.md:\n"
        "  python research/db/load_1m_roostoo_universe.py --db data/research_roostoo_1m.duckdb\n"
    )

con = duckdb.connect(str(DB_FILE), read_only=True)
print("Repo:", ROOT)
print("DuckDB:", DB_FILE)

In [ ]:
# Universe check: expect ~67 distinct bases (Roostoo USD pairs)
EXPECTED_TOKENS = 67

intervals_avail = con.execute(
    "SELECT interval, COUNT(*)::BIGINT AS n FROM ohlcv GROUP BY 1 ORDER BY 1"
).df()
display(intervals_avail)

# Use 1h for universe plots (manageable rows); change if you only loaded 1m
ITV = "1h"
if ITV not in set(intervals_avail["interval"]):
    ITV = intervals_avail.sort_values("n", ascending=False).iloc[0]["interval"]
    print(f"Fallback ITV = {ITV!r} (1h not found)")

symbols = con.execute(
    "SELECT DISTINCT symbol FROM ohlcv WHERE interval = ? ORDER BY symbol",
    [ITV],
).df()["symbol"].tolist()

n_sym = len(symbols)
print(f"\nInterval: {ITV} | distinct symbols: {n_sym} (expected ~{EXPECTED_TOKENS})")
if n_sym < EXPECTED_TOKENS - 2:
    print(
        "⚠ Fewer symbols than full Roostoo universe — re-run the loader without --limit, "
        "or check roostoo_load_meta / failed downloads."
    )
symbols[:8], "...", symbols[-3:]

In [ ]:
# Per-symbol coverage (same interval as universe EDA)
coverage = con.execute(
    """
    SELECT symbol,
           COUNT(*)::BIGINT AS n_rows,
           MIN(ts) AS min_ts,
           MAX(ts) AS max_ts
    FROM ohlcv
    WHERE interval = ?
    GROUP BY 1
    ORDER BY symbol
    """,
    [ITV],
).df()
coverage["min_ts"] = pd.to_datetime(coverage["min_ts"])
coverage["max_ts"] = pd.to_datetime(coverage["max_ts"])
display(coverage)
print(f"Total rows ({ITV}): {coverage['n_rows'].sum():,}")

In [ ]:
# Wide close matrix: all symbols, aligned timestamps
long_px = con.execute(
    "SELECT ts, symbol, close, volume FROM ohlcv WHERE interval = ? ORDER BY ts, symbol",
    [ITV],
).df()
long_px["ts"] = pd.to_datetime(long_px["ts"])

close_w = long_px.pivot(index="ts", columns="symbol", values="close").sort_index()
vol_w = long_px.pivot(index="ts", columns="symbol", values="volume").sort_index()

print(close_w.shape, "(timesteps × symbols)")
close_w.iloc[:3, :6]

In [ ]:
# Return matrix construction (coverage-aware, avoids truncating to shortest symbol history)
#
# Issue we observed: some symbols stop at 2026-02-28 in 1h, while others extend later.
# If we require 100% symbol coverage at each timestamp, EDA appears to "stop" at Feb 28.
#
# Fix: forward-fill prices per symbol, then keep timestamps that have at least `min_frac` symbols.

min_frac = 0.95  # keep timestamps where >=95% of symbols have data

px_ff = close_w.sort_index().ffill()
valid_frac = px_ff.notna().mean(axis=1)

# Keep a long sample window (doesn't force the shortest-history intersection)
px = px_ff.loc[valid_frac >= min_frac]

rets = px.pct_change()
rets = rets.loc[rets.notna().mean(axis=1) >= min_frac]

# Drop symbols that are entirely missing (shouldn't happen after ffill, but keep safe)
rets = rets.dropna(axis=1, how="all")

print(
    "Return sample:",
    rets.shape,
    "|",
    rets.index.min(),
    "→",
    rets.index.max(),
    f"| min_frac={min_frac}"
)

bars_per_year = {
    "1h": 24 * 365,
    "1m": 24 * 60 * 365,
    "5m": 12 * 24 * 365,
    "15m": 4 * 24 * 365,
}.get(ITV, 24 * 365)

summary = pd.DataFrame(
    {
        "mean_ret": rets.mean(),
        "vol": rets.std(),
        "sharpe": rets.mean() / rets.std() * np.sqrt(bars_per_year),
        "n": rets.notna().sum(),
    }
).sort_values("sharpe", ascending=False)

display(summary.head(15))
display(summary.tail(5))


In [ ]:
# Universe distribution: Sharpe & volatility
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(summary["sharpe"].dropna(), bins=20, color="steelblue", edgecolor="white")
axes[0].set_title(f"Cross-sectional Sharpe ({ITV}, annualized)")
axes[0].axvline(0, color="red", linestyle="--", linewidth=1)

axes[1].hist(summary["vol"].dropna(), bins=20, color="coral", edgecolor="white")
axes[1].set_title(f"Cross-sectional volatility ({ITV}, per-bar)")
plt.tight_layout()
plt.show()

In [ ]:
# Full-universe correlation matrix (no annotations — too many names)
corr = rets.corr()
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    corr,
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    xticklabels=True,
    yticklabels=True,
    ax=ax,
)
ax.set_title(f"Return correlation — all symbols ({ITV})")
plt.xticks(rotation=90, fontsize=6)
plt.yticks(rotation=0, fontsize=6)
plt.tight_layout()
plt.show()

mask = np.triu(np.ones(corr.shape), k=1).astype(bool)
upper = corr.where(mask).stack()
print(f"Mean pairwise corr: {upper.mean():.3f} | median: {upper.median():.3f}")

In [ ]:
# Equal-weight universe index (normalized to 1)
ew = (1 + rets).cumprod()
ew["EW_PORTFOLIO"] = ew.mean(axis=1)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(ew.index, ew["EW_PORTFOLIO"], label="Equal-weight (all)", color="black", linewidth=1.5)
for c in ["BTC-USD", "ETH-USD", "SOL-USD"]:
    if c in ew.columns:
        ax.plot(ew.index, ew[c], alpha=0.35, linewidth=1, label=c)
ax.set_title(f"Cumulative return (1 = start) — {ITV}")
ax.legend(loc="upper left", fontsize=8, ncol=2)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Average log volume by symbol (universe cross-section)
v = vol_w.reindex(px.index).dropna(how="all")
lv = np.log1p(v)
mean_lv = lv.mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, max(4, len(mean_lv) * 0.12)))
mean_lv.iloc[::-1].plot(kind="barh", ax=ax, color="slategray")
ax.set_title(f"Mean log(1+volume) by symbol ({ITV})")
plt.tight_layout()
plt.show()

In [ ]:
# Optional: duckdb.connect is cheap; restart kernel if you change DB on disk.
# con.close()

In [ ]:
# Build technical feature set for 67-token universe from 1h returns (correlation/PCA-style + distributional stats)
import numpy as np
import pandas as pd

# Assumes con, ITV, close_w, px, rets exist from earlier cells in this notebook state
assert 'rets' in globals(), "Expected `rets` returns matrix to exist (run cells 5-6)."

rets_1h = rets.copy()

# --- (A) Return distribution features (interpretable) ---
tech_feat = pd.DataFrame(index=rets_1h.columns)
tech_feat["mean_ret"] = rets_1h.mean()
tech_feat["vol"] = rets_1h.std()
tech_feat["skew"] = rets_1h.skew()
tech_feat["kurt"] = rets_1h.kurtosis()
tech_feat["p01"] = rets_1h.quantile(0.01)
tech_feat["p99"] = rets_1h.quantile(0.99)
tech_feat["tail_1pct"] = tech_feat["p99"] - tech_feat["p01"]

# --- (B) Correlation-to-market and beta-to-market (equal-weight index) ---
mkt = rets_1h.mean(axis=1)  # equal-weight "market" return
tech_feat["corr_to_mkt"] = rets_1h.apply(lambda s: s.corr(mkt))
tech_feat["beta_to_mkt"] = rets_1h.apply(lambda s: s.cov(mkt) / mkt.var())

# --- (C) First principal-component exposure (no sklearn): eigenvector of corr matrix ---
C = rets_1h.corr().values
# Guard numerical issues
w, v = np.linalg.eigh(C)
pc1 = v[:, -1]
pc1 = pc1 / np.linalg.norm(pc1)
tech_feat["pc1_loading"] = pc1
tech_feat["pc1_abs_loading"] = np.abs(pc1)

# --- (D) Simple momentum proxy: last 7d vs last 30d cumulative return (1h bars) ---
bars_7d = 7 * 24
bars_30d = 30 * 24
cum = (1 + rets_1h).cumprod()
tech_feat["mom_7d"] = (cum.iloc[-1] / cum.iloc[-min(bars_7d, len(cum))] - 1).replace([np.inf, -np.inf], np.nan)
tech_feat["mom_30d"] = (cum.iloc[-1] / cum.iloc[-min(bars_30d, len(cum))] - 1).replace([np.inf, -np.inf], np.nan)

# Clean
tech_feat = tech_feat.replace([np.inf, -np.inf], np.nan).dropna()
print("Technical feature matrix:", tech_feat.shape)
display(tech_feat.head())

# Save for downstream clustering/classification
tech_feat.to_csv("outputs/eda/tech_features_67.csv")


In [ ]:
# Build a purely-technical token classification using correlation + simple distributional features
# Produces interpretable cluster labels for the 67-token universe.

from pathlib import Path

assert 'rets_1h' in globals(), "Expected rets_1h from earlier cells"
assert 'tech_feat' in globals(), "Expected tech_feat feature table from cell 12"

# (1) Correlation-distance matrix
corr_67 = rets_1h.corr()
dist = 1 - corr_67

# (2) Build a compact feature set for clustering (all interpretable)
feat_cols = [
    'vol', 'skew', 'kurt', 'tail_1pct', 'corr_to_mkt', 'beta_to_mkt',
    'pc1_abs_loading', 'mom_30d'
]
X = tech_feat[feat_cols].copy()

# Standardize (z-score) without sklearn
Xz = (X - X.mean()) / X.std(ddof=0)
Xz = Xz.replace([np.inf, -np.inf], np.nan).dropna()

# Keep correlation matrix aligned with tokens that survived feature cleaning
common = Xz.index.intersection(corr_67.index)
Xz = Xz.loc[common]
dist = dist.loc[common, common]

print("Clustering universe size:", len(common))

# (3) K-means (simple, interpretable centroid summaries) without sklearn
# We'll use a tiny implementation for k in [3..8] and pick based on inertia elbow.

def kmeans_numpy(X: np.ndarray, k: int, n_init: int = 25, n_iter: int = 200, seed: int = 0):
    rng = np.random.default_rng(seed)
    best = None
    best_inertia = np.inf

    for i in range(n_init):
        # k-means++-ish init: sample points uniformly for simplicity
        idx = rng.choice(X.shape[0], size=k, replace=False)
        centers = X[idx].copy()

        for _ in range(n_iter):
            # assign
            d2 = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
            labels = d2.argmin(axis=1)
            # update
            new_centers = np.vstack([
                X[labels == j].mean(axis=0) if np.any(labels == j) else centers[j]
                for j in range(k)
            ])
            if np.allclose(new_centers, centers, atol=1e-6, rtol=0):
                centers = new_centers
                break
            centers = new_centers

        inertia = ((X - centers[labels]) ** 2).sum()
        if inertia < best_inertia:
            best_inertia = inertia
            best = (labels.copy(), centers.copy(), inertia)

    return best  # labels, centers, inertia

X_arr = Xz.values
ks = list(range(3, 9))
results = []
for k in ks:
    labels, centers, inertia = kmeans_numpy(X_arr, k=k, seed=42)
    results.append((k, inertia))

inertia_df = pd.DataFrame(results, columns=['k', 'inertia'])
display(inertia_df)

# Choose a default k=5 (good balance for 67); user can adjust
k_default = 5
labels, centers, inertia = kmeans_numpy(X_arr, k=k_default, seed=42)

tech_labels = pd.DataFrame({
    'symbol': common,
    'tech_cluster': labels,
}).set_index('symbol')

# Summarize clusters in original (non-z) units for interpretability
cluster_summary = X.loc[common].join(tech_labels).groupby('tech_cluster').agg(['mean', 'median', 'count'])

print(f"\nTechnical clusters (k={k_default}) summary:")
display(cluster_summary)

# Save outputs
out_dir = Path('outputs/eda')
out_dir.mkdir(parents=True, exist_ok=True)
tech_labels.to_csv(out_dir / f'tech_clusters_k{k_default}.csv')
inertia_df.to_csv(out_dir / 'kmeans_inertia_scan.csv', index=False)

# Quick view: list tokens per cluster
for cl in sorted(tech_labels['tech_cluster'].unique()):
    members = tech_labels.index[tech_labels['tech_cluster'] == cl].tolist()
    print(f"Cluster {cl} (n={len(members)}):", ", ".join(members))


In [ ]:
# Improve CoinGecko fundamentals coverage by building a robust symbol->id map
# UPDATED: uses on-disk caching (cached_get_json) + gentle backoff to avoid rate limits.

import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests

assert 'tech_feat' in globals(), "Run technical features cell first (tech_feat)."
assert 'cached_get_json' in globals(), "`cached_get_json` is defined in the imports cell — re-run that cell if needed."

symbols_67 = list(tech_feat.index)
base_tickers = sorted({s.split('-')[0].upper() for s in symbols_67})

BASE = "https://api.coingecko.com/api/v3"
session = requests.Session()
session.headers.update({"accept": "application/json"})

CG_TTL_SECONDS = 24 * 3600  # cache fundamentals for 24h


def cg_get(url: str, params: dict | None = None, *, ttl: int = CG_TTL_SECONDS, tries: int = 4):
    """Cached GET with simple backoff on rate limit / transient errors."""
    last_exc = None
    for attempt in range(tries):
        try:
            data, meta = cached_get_json(
                url,
                params=params or {},
                session=session,
                timeout=60,
                ttl_seconds=ttl,
                cache_subdir="coingecko",
            )
            # If CoinGecko returns a rate-limit JSON payload with a 429 status code,
            # cached_get_json would have raised before caching. So we handle 429 by retrying here
            # only if it came back as a normal 200 payload (rare), otherwise exceptions are handled below.
            return data, meta
        except requests.HTTPError as e:
            last_exc = e
            # Backoff on 429 or 5xx
            status = getattr(e.response, "status_code", None)
            if status in (429, 500, 502, 503, 504):
                sleep_s = min(60, (2 ** attempt) * 5)
                print(f"CoinGecko HTTP {status}; sleeping {sleep_s}s then retrying...")
                time.sleep(sleep_s)
                continue
            raise
        except Exception as e:
            last_exc = e
            sleep_s = min(60, (2 ** attempt) * 5)
            print(f"CoinGecko error {type(e).__name__}; sleeping {sleep_s}s then retrying...")
            time.sleep(sleep_s)

    raise RuntimeError(f"CoinGecko request failed after {tries} tries: {url} | last={last_exc!r}")


# 1) Download full coin list (id, symbol, name) once (cached)
coin_list, meta = cg_get(f"{BASE}/coins/list", params={"include_platform": "false"}, ttl=7 * 24 * 3600)
coins_df = pd.DataFrame(coin_list if isinstance(coin_list, list) else [])
if coins_df.empty or "symbol" not in coins_df.columns:
    raise RuntimeError(
        "CoinGecko /coins/list did not return a usable list (likely rate-limited or blocked). "
        "Try rerunning later; cached_get_json will help once one successful pull happens."
    )
coins_df["symbol"] = coins_df["symbol"].astype(str).str.upper()
coins_df["name"] = coins_df["name"].astype(str)

# 2) Candidate ids by symbol (many-to-one); keep all candidates for later disambiguation
sym_to_ids = coins_df.groupby("symbol")["id"].apply(list).to_dict()

# 3) Heuristic id pick: first candidate; record ambiguity (we can refine later)
rows = []
ambiguous = []
missing = []
for tkr in base_tickers:
    ids = sym_to_ids.get(tkr)
    if not ids:
        missing.append(tkr)
        continue
    if len(ids) > 1:
        ambiguous.append((tkr, ids[:10]))
    rows.append({"ticker": tkr, "cg_id": ids[0], "n_candidates": len(ids)})

id_map = pd.DataFrame(rows).set_index("ticker").sort_index()
print(
    f"Tickers requested: {len(base_tickers)} | mapped: {len(id_map)} | "
    f"missing: {len(missing)} | ambiguous: {len(ambiguous)}"
)
if missing:
    print("Missing tickers (no CG symbol match):", missing)
if ambiguous:
    print("Ambiguous tickers (showing first few ids):")
    for tkr, ids in ambiguous[:15]:
        print(f"  {tkr}: {ids}")

display(id_map.head())

# 4) Fetch fundamentals for mapped ids (cached)
fund_rows = []
errors = []
for i, (tkr, r) in enumerate(id_map.iterrows(), start=1):
    coin_id = r["cg_id"]
    try:
        j, meta = cg_get(
            f"{BASE}/coins/{coin_id}",
            params={
                "localization": "false",
                "tickers": "false",
                "market_data": "true",
                "community_data": "false",
                "developer_data": "false",
                "sparkline": "false",
            },
            ttl=CG_TTL_SECONDS,
        )
        md = j.get("market_data", {})
        cats = j.get("categories", []) or []
        fund_rows.append(
            {
                "ticker": tkr,
                "cg_id": coin_id,
                "name": j.get("name"),
                "market_cap_usd": (md.get("market_cap") or {}).get("usd"),
                "fdv_usd": (md.get("fully_diluted_valuation") or {}).get("usd"),
                "circulating_supply": md.get("circulating_supply"),
                "total_supply": md.get("total_supply"),
                "max_supply": md.get("max_supply"),
                "categories": ";".join(cats),
                "n_categories": len(cats),
            }
        )
        # Only sleep when we actually hit the network
        if not meta.get("cached", False):
            time.sleep(0.6)
    except Exception as e:
        errors.append((tkr, coin_id, repr(e)))

fund2 = pd.DataFrame(fund_rows).set_index("ticker").sort_index()
print(f"Fundamentals fetched: {fund2.shape} | errors: {len(errors)}")
if errors:
    print("First 10 errors:", errors[:10])

display(fund2.head())

# 5) Join back to our symbol universe and build numeric features
ticker_to_symbol = {s.split('-')[0].upper(): s for s in symbols_67}
fund_symbol = fund2.copy()
fund_symbol["symbol"] = fund_symbol.index.map(ticker_to_symbol)
fund_symbol = fund_symbol.dropna(subset=["symbol"]).set_index("symbol")

fund_feat2 = pd.DataFrame(index=fund_symbol.index)
fund_feat2["mcap_log"] = np.log1p(pd.to_numeric(fund_symbol["market_cap_usd"], errors="coerce"))
fund_feat2["fdv_log"] = np.log1p(pd.to_numeric(fund_symbol["fdv_usd"], errors="coerce"))
with np.errstate(divide="ignore", invalid="ignore"):
    fund_feat2["fdv_to_mcap"] = (
        pd.to_numeric(fund_symbol["fdv_usd"], errors="coerce")
        / pd.to_numeric(fund_symbol["market_cap_usd"], errors="coerce")
    ).replace([np.inf, -np.inf], np.nan)
fund_feat2["n_categories"] = pd.to_numeric(fund_symbol["n_categories"], errors="coerce")

print("Fund feature coverage vs 67 symbols:", fund_feat2.shape[0], "/", len(symbols_67))

# Save artifacts
out_dir = Path("outputs/eda")
out_dir.mkdir(parents=True, exist_ok=True)
id_map.to_csv(out_dir / "coingecko_symbol_id_map.csv")
fund2.to_csv(out_dir / "fundamentals_coingecko_by_ticker.csv")
fund_feat2.to_csv(out_dir / "fundamentals_features_coingecko.csv")
print("Saved:")
print(" -", out_dir / "coingecko_symbol_id_map.csv")
print(" -", out_dir / "fundamentals_coingecko_by_ticker.csv")
print(" -", out_dir / "fundamentals_features_coingecko.csv")


## Token classification (technical + CoinGecko fundamentals)

**Technical** — From DuckDB returns only: vol, tails, market beta, PC1, momentum, k-means clusters (`outputs/eda/tech_clusters_k5.csv`).

**Fundamentals** — **CoinGecko** public API only (`api.coingecko.com`): mcap, FDV, categories; cached on disk under `outputs/api_cache/coingecko/`.

No second market-data API (e.g. CoinMarketCap) — one pipeline, easier rate-limit and debugging.

Next cells: technical type table → fundamentals features file → unified table.


In [ ]:
# Build a final, interpretable technical "type" table for the 67-token universe
# - Uses existing tech_clusters (k=5), correlation structure, and a lightweight cointegration proxy
# - Saves an artifact you can use downstream (no backtesting)

from pathlib import Path
import numpy as np
import pandas as pd

assert 'rets_1h' in globals() and 'px' in globals(), "Run earlier cells to create `rets_1h` and `px`."
assert 'tech_feat' in globals(), "Expected `tech_feat` (cell 12)."
assert 'tech_labels' in globals(), "Expected `tech_labels` from technical clustering (cell 13)."

out_dir = Path('outputs/eda')
out_dir.mkdir(parents=True, exist_ok=True)

# --- 1) Basic cluster + key exposures ---
types = tech_feat.join(tech_labels, how='inner')

# Give clusters a human-friendly descriptor based on medians
cluster_profile = (
    types.groupby('tech_cluster')
    .agg({
        'vol': 'median',
        'tail_1pct': 'median',
        'beta_to_mkt': 'median',
        'corr_to_mkt': 'median',
        'mom_30d': 'median',
    })
)

def describe_cluster(row):
    vol = row['vol']
    tail = row['tail_1pct']
    beta = row['beta_to_mkt']
    mom = row['mom_30d']

    vol_tag = 'low-vol' if vol < types['vol'].median() else 'high-vol'
    tail_tag = 'fat-tail' if tail > types['tail_1pct'].median() else 'thin-tail'
    beta_tag = 'high-beta' if beta > 1.0 else 'low-beta'
    mom_tag = 'pos-mom' if mom > 0 else 'neg-mom'
    return f"{vol_tag}, {tail_tag}, {beta_tag}, {mom_tag}"

cluster_desc = cluster_profile.apply(describe_cluster, axis=1).to_dict()
types['tech_type'] = types['tech_cluster'].map(cluster_desc)

# --- 2) Correlation peers (top 3) ---
corr_ = rets_1h.corr()
peers = {}
for s in corr_.columns:
    top = corr_[s].drop(index=s).sort_values(ascending=False).head(3)
    peers[s] = ";".join([f"{i}:{v:.2f}" for i, v in top.items()])

types['top_corr_peers'] = pd.Series(peers)

# --- 3) Lightweight cointegration proxy: log-price spread stability vs top peer ---
# For each token, take its top correlated peer; compute hedge ratio via OLS on log prices;
# then compute spread std relative to each leg's std.
# Lower ratio => more "spread-stable" relationship (cointegration-like, proxy only).
logp = np.log(px.replace(0, np.nan)).ffill().dropna(how='any')

proxy_rows = []
for s in logp.columns:
    peer = corr_[s].drop(index=s).idxmax()
    x = logp[peer].values
    y = logp[s].values

    # OLS slope b = cov(x,y)/var(x)
    vx = np.var(x)
    if vx == 0 or not np.isfinite(vx):
        continue
    b = np.cov(x, y, ddof=0)[0, 1] / vx
    spread = y - b * x

    spread_std = np.std(spread)
    y_std = np.std(y)
    x_std = np.std(x)

    stability_ratio = spread_std / (y_std + 1e-12)  # normalize by token volatility in log-space

    proxy_rows.append({
        'symbol': s,
        'proxy_peer': peer,
        'proxy_hedge_ratio': b,
        'proxy_spread_std': spread_std,
        'proxy_spread_stability': stability_ratio,
    })

proxy = pd.DataFrame(proxy_rows).set_index('symbol')

types = types.join(proxy, how='left')

# --- 4) Final ordering & save ---
cols = [
    'tech_cluster', 'tech_type',
    'vol', 'tail_1pct', 'skew', 'kurt',
    'corr_to_mkt', 'beta_to_mkt', 'pc1_abs_loading',
    'mom_7d', 'mom_30d',
    'top_corr_peers',
    'proxy_peer', 'proxy_hedge_ratio', 'proxy_spread_stability',
]
cols = [c for c in cols if c in types.columns]

tech_types = types[cols].sort_values(['tech_cluster', 'vol'], ascending=[True, False])

display(tech_types.head(12))
print("\nTechnical type table shape:", tech_types.shape)

tech_types.to_csv(out_dir / 'technical_token_types.csv')
print("Saved:", out_dir / 'technical_token_types.csv')

# Also save cluster descriptors
pd.Series(cluster_desc, name='cluster_descriptor').to_csv(out_dir / 'tech_cluster_descriptors.csv')
print("Saved:", out_dir / 'tech_cluster_descriptors.csv')


In [ ]:
# Fundamentals clustering (CoinGecko only — uses `fund_feat2` from the prior cell)
from pathlib import Path
import numpy as np
import pandas as pd

assert "fund_feat2" in globals(), "Run the CoinGecko /coins/list fundamentals cell first (produces fund_feat2)."
assert "kmeans_numpy" in globals(), "Expected kmeans_numpy from technical clustering cell."
assert "tech_feat" in globals()

out_dir = Path("outputs/eda")
out_dir.mkdir(parents=True, exist_ok=True)

universe = tech_feat.index.tolist()
fund_feat_final = fund_feat2.reindex(universe).copy()
fund_cols = ["mcap_log", "fdv_log", "fdv_to_mcap", "n_categories"]
fund_feat_final = fund_feat_final[fund_cols]
fund_feat_final.to_csv(out_dir / "fundamentals_features.csv")
print("Saved:", out_dir / "fundamentals_features.csv")

Xf = fund_feat_final.dropna(how="any")
coverage = len(Xf)
print(f"Fundamentals complete rows: {coverage} / {len(universe)}")

fund_labels = pd.DataFrame(index=fund_feat_final.index, columns=["fund_cluster"])
fund_labels["fund_cluster"] = fund_labels["fund_cluster"].astype("object")
fund_cluster_summary = None

if coverage < 10:
    print("Coverage low — skipping k-means; optional mcap buckets for rows that exist.")
    if coverage >= 1:
        qs = Xf["mcap_log"].quantile([0.33, 0.66])

        def mcap_bucket(v):
            if v <= qs.iloc[0]:
                return "mcap_small"
            if v <= qs.iloc[1]:
                return "mcap_mid"
            return "mcap_large"

        fund_labels.loc[Xf.index, "fund_cluster"] = Xf["mcap_log"].apply(mcap_bucket)
else:
    Xfz = (Xf - Xf.mean()) / Xf.std(ddof=0)
    Xfz = Xfz.replace([np.inf, -np.inf], np.nan).dropna()
    k_fund = 3 if Xfz.shape[0] < 25 else 5
    labels, centers, inertia = kmeans_numpy(Xfz.values, k=k_fund, seed=42)
    fund_labels.loc[Xfz.index, "fund_cluster"] = labels
    fund_cluster_summary = (
        Xf.loc[Xfz.index]
        .join(fund_labels.loc[Xfz.index])
        .groupby("fund_cluster")
        .agg(["median", "count"])
    )
    print(f"Fundamentals clusters (k={k_fund}):")
    display(fund_cluster_summary)
    fund_labels.loc[Xfz.index].to_csv(out_dir / f"fund_clusters_k{k_fund}.csv")
    print("Saved:", out_dir / f"fund_clusters_k{k_fund}.csv")

for cl in sorted(pd.unique(fund_labels["fund_cluster"].dropna())):
    members = fund_labels.index[fund_labels["fund_cluster"] == cl].tolist()
    print(f"Fund {cl} (n={len(members)}):", ", ".join(members[:30]) + ("..." if len(members) > 30 else ""))


## Classification artifacts

**Technical** (always): `technical_token_types.csv`, `tech_clusters_k5.csv`, `tech_cluster_descriptors.csv`

**Fundamentals** (CoinGecko): `fundamentals_features.csv` — coverage depends on API; re-run after cache warms up.

**Unified**: `unified_token_classification.csv` combines technical clusters with fundamentals buckets/clusters when present.

~67 tokens → prefer clustering / linear models over neural nets.


In [ ]:
# Display final classification artifacts produced by the notebook (technical + fundamentals)
from pathlib import Path
import pandas as pd

out_dir = Path('outputs/eda')

tech_types_path = out_dir / 'technical_token_types.csv'
tech_clusters_path = out_dir / 'tech_clusters_k5.csv'
fund_feat_path = out_dir / 'fundamentals_features.csv'

print('--- Artifacts present ---')
for p in [tech_types_path, tech_clusters_path, fund_feat_path]:
    print(f"{p}: {'OK' if p.exists() else 'MISSING'}")

if tech_types_path.exists():
    tech_types_df = pd.read_csv(tech_types_path, index_col=0)
    display(tech_types_df.head(10))
    print('\nTechnical clusters:')
    display(tech_types_df['tech_cluster'].value_counts().sort_index().rename('n_tokens'))

if tech_clusters_path.exists():
    tech_clusters_df = pd.read_csv(tech_clusters_path, index_col=0)
    # quick member listing per cluster
    print('\nCluster members (alphabetical):')
    for cl, g in tech_clusters_df.sort_index().groupby('tech_cluster'):
        members = g.index.tolist()
        print(f"  Cluster {cl} (n={len(members)}): {', '.join(members)}")

if fund_feat_path.exists():
    fund_feat_df = pd.read_csv(fund_feat_path, index_col=0)
    coverage = fund_feat_df['mcap_log'].notna().sum() if 'mcap_log' in fund_feat_df.columns else fund_feat_df.notna().any(axis=1).sum()
    print(f"\nFundamentals feature coverage: {coverage} / {fund_feat_df.shape[0]} symbols")
    display(fund_feat_df.head(10))


In [ ]:
# Build a unified (technical + fundamentals) feature table with coverage diagnostics
from pathlib import Path
import numpy as np
import pandas as pd

assert 'tech_feat' in globals(), "Expected tech_feat (technical features)."

out_dir = Path('outputs/eda')
out_dir.mkdir(parents=True, exist_ok=True)

# Load latest technical type table if present (for convenience)
tech_types_path = out_dir / 'technical_token_types.csv'
tech_types_df = pd.read_csv(tech_types_path, index_col=0) if tech_types_path.exists() else None

# Load fundamentals features (may be partial)
fund_path = out_dir / 'fundamentals_features.csv'
fund_df = pd.read_csv(fund_path, index_col=0) if fund_path.exists() else pd.DataFrame()

# Define columns to use from each side (keep interpretable + avoid leakage)
tech_cols = [
    'vol','skew','kurt','tail_1pct','corr_to_mkt','beta_to_mkt','pc1_abs_loading','mom_30d'
]
tech_cols = [c for c in tech_cols if c in tech_feat.columns]

fund_cols = ['mcap_log','fdv_log','fdv_to_mcap','n_categories']
fund_cols = [c for c in fund_cols if c in fund_df.columns]

X_tech = tech_feat[tech_cols].copy()
X_fund = fund_df[fund_cols].copy() if not fund_df.empty else pd.DataFrame(index=X_tech.index)

# Align universe
universe = X_tech.index.union(X_fund.index)
X_tech = X_tech.reindex(universe)
X_fund = X_fund.reindex(universe)

# Coverage diagnostics
n_total = len(universe)
tech_cov = X_tech.notna().all(axis=1).sum()
fund_cov = X_fund.notna().all(axis=1).sum() if len(fund_cols) else 0
both_cov = (X_tech.notna().all(axis=1) & X_fund.notna().all(axis=1)).sum() if len(fund_cols) else 0

print(f"Universe size (union): {n_total}")
print(f"Technical complete rows: {tech_cov}/{n_total}")
print(f"Fundamentals complete rows: {fund_cov}/{n_total} (cols={fund_cols})")
print(f"Complete on both: {both_cov}/{n_total}")

# Unified feature matrix (keep both sets side-by-side)
X_all = pd.concat([X_tech.add_prefix('tech__'), X_fund.add_prefix('fund__')], axis=1)

# Add provenance flags
prov = pd.DataFrame(index=universe)
prov['has_tech'] = X_tech.notna().all(axis=1)
prov['has_fund'] = X_fund.notna().all(axis=1) if len(fund_cols) else False
prov['has_both'] = prov['has_tech'] & prov['has_fund']

# Save for downstream steps
X_all.to_csv(out_dir / 'unified_features_tech_fund.csv')
prov.to_csv(out_dir / 'unified_features_provenance.csv')

print("Saved:", out_dir / 'unified_features_tech_fund.csv')
print("Saved:", out_dir / 'unified_features_provenance.csv')

display(X_all.head())
display(prov.value_counts().rename('n_tokens'))


In [ ]:
# Build a unified token classification using BOTH technical + fundamentals where available
# - Always assigns a technical cluster/type (full coverage)
# - Assigns a fundamentals cluster when fundamentals are present
# - Creates a combined label that refines technical type with fundamentals bucket

from pathlib import Path
import numpy as np
import pandas as pd

out_dir = Path('outputs/eda')
out_dir.mkdir(parents=True, exist_ok=True)

# Inputs expected from prior cells
assert 'tech_types_df' in globals() or (out_dir / 'technical_token_types.csv').exists(), "Missing technical type table"
assert (out_dir / 'fundamentals_features.csv').exists(), "Missing fundamentals_features.csv (even if partial)"

tech_types = (
    tech_types_df.copy()
    if 'tech_types_df' in globals() and isinstance(tech_types_df, pd.DataFrame)
    else pd.read_csv(out_dir / 'technical_token_types.csv', index_col=0)
)
fund_feat = pd.read_csv(out_dir / 'fundamentals_features.csv', index_col=0)

# Determine which rows have complete fundamentals (for a stable fundamentals cluster assignment)
fund_cols = [c for c in ['mcap_log','fdv_log','fdv_to_mcap','n_categories'] if c in fund_feat.columns]
fund_complete = fund_feat[fund_cols].notna().all(axis=1) if fund_cols else pd.Series(False, index=fund_feat.index)

# Cluster fundamentals ONLY on complete rows (avoid noisy partial clustering)
# Use object dtype because we may store either numeric cluster ids or string buckets
fund_labels = pd.DataFrame(index=fund_feat.index, columns=['fund_cluster'])
fund_labels['fund_cluster'] = fund_labels['fund_cluster'].astype('object')
fund_cluster_summary = None

if fund_complete.sum() >= 10 and 'kmeans_numpy' in globals():
    Xf = fund_feat.loc[fund_complete, fund_cols].astype(float).copy()
    Xfz = (Xf - Xf.mean()) / Xf.std(ddof=0)
    Xfz = Xfz.replace([np.inf, -np.inf], np.nan).dropna()

    k_fund = 3 if Xfz.shape[0] < 25 else 5
    labels, centers, inertia = kmeans_numpy(Xfz.values, k=k_fund, seed=42)
    fund_labels.loc[Xfz.index, 'fund_cluster'] = labels

    fund_cluster_summary = (
        Xf.loc[Xfz.index]
        .join(fund_labels.loc[Xfz.index])
        .groupby('fund_cluster')
        .agg(['median', 'count'])
    )
else:
    # If fundamentals coverage is too low, we still keep fundamentals features, but no clustering.
    # We'll create coarse, interpretable buckets for those few tokens that do have fundamentals.
    if fund_complete.sum() >= 1:
        x = fund_feat.loc[fund_complete, fund_cols].copy()
        # simple buckets for mcap_log (small/med/large by quantiles within available set)
        qs = x['mcap_log'].quantile([0.33, 0.66]) if 'mcap_log' in x.columns else None
        if qs is not None:
            def mcap_bucket(v):
                if v <= qs.iloc[0]:
                    return 'mcap_small'
                elif v <= qs.iloc[1]:
                    return 'mcap_mid'
                else:
                    return 'mcap_large'
            fund_labels.loc[x.index, 'fund_cluster'] = x['mcap_log'].apply(mcap_bucket)

# Build combined classification table
combined = tech_types.copy()
combined = combined.join(fund_feat.add_prefix('fund__'), how='left')
combined = combined.join(fund_labels, how='left')

combined['has_fundamentals'] = fund_complete.reindex(combined.index).fillna(False)

# Combined label logic:
# - Base: technical type string (cluster descriptor)
# - If fundamentals label exists: append it to create a refined combined type
combined['combined_type'] = combined['tech_type'].astype(str)
mask = combined['fund_cluster'].notna()
combined.loc[mask, 'combined_type'] = combined.loc[mask].apply(
    lambda r: f"{r['tech_type']} | fund:{r['fund_cluster']}", axis=1
)

# Human-readable compact label
combined['combined_label'] = combined.apply(
    lambda r: f"T{int(r['tech_cluster'])}" + (f"-F{r['fund_cluster']}" if pd.notna(r['fund_cluster']) else ""),
    axis=1
)

# Save artifacts
combined_out = out_dir / 'unified_token_classification.csv'
combined.to_csv(combined_out)
print('Saved:', combined_out)

# Coverage report
print('Universe size:', combined.shape[0])
print('Technical coverage:', combined['tech_cluster'].notna().sum(), '/', combined.shape[0])
print('Fundamentals complete coverage:', int(combined['has_fundamentals'].sum()), '/', combined.shape[0])

# Show a compact view
show_cols = [
    'tech_cluster','tech_type','fund_cluster','has_fundamentals','combined_label','combined_type'
]
show_cols = [c for c in show_cols if c in combined.columns]

display(combined[show_cols].sort_values(['tech_cluster','has_fundamentals'], ascending=[True, False]).head(25))

if fund_cluster_summary is not None:
    print('\nFundamentals cluster summary:')
    display(fund_cluster_summary)
else:
    print('\nFundamentals clustering not run (insufficient fundamentals coverage). Using coarse buckets if any fundamentals are present.')


In [ ]:
# Diagnose why data appears to stop after March 1 (check max timestamps per interval/symbol)
import pandas as pd

assert 'con' in globals(), "Expected DuckDB connection `con` (run cell 2)."

# 1) Overall max ts per interval
max_by_interval = con.execute(
    """
    SELECT interval,
           MIN(ts) AS min_ts,
           MAX(ts) AS max_ts,
           COUNT(*)::BIGINT AS n_rows,
           COUNT(DISTINCT symbol) AS n_symbols
    FROM ohlcv
    GROUP BY 1
    ORDER BY 1
    """
).df()
max_by_interval['min_ts'] = pd.to_datetime(max_by_interval['min_ts'])
max_by_interval['max_ts'] = pd.to_datetime(max_by_interval['max_ts'])
print("=== Coverage by interval ===")
display(max_by_interval)

# 2) Max ts per symbol for the interval used in EDA
ITV_USED = globals().get('ITV', '1h')
max_by_symbol = con.execute(
    """
    SELECT symbol,
           MIN(ts) AS min_ts,
           MAX(ts) AS max_ts,
           COUNT(*)::BIGINT AS n_rows
    FROM ohlcv
    WHERE interval = ?
    GROUP BY 1
    ORDER BY 2, 1
    """,
    [ITV_USED],
).df()
max_by_symbol['min_ts'] = pd.to_datetime(max_by_symbol['min_ts'])
max_by_symbol['max_ts'] = pd.to_datetime(max_by_symbol['max_ts'])

print(f"\n=== Coverage by symbol (interval={ITV_USED}) ===")
display(max_by_symbol)

# 3) Specifically check if there are ANY rows on/after 2026-03-01
cut = pd.Timestamp('2026-03-01')
after_cut = con.execute(
    """
    SELECT interval, COUNT(*)::BIGINT AS n
    FROM ohlcv
    WHERE ts >= ?
    GROUP BY 1
    ORDER BY 1
    """,
    [cut],
).df()
print("\n=== Rows on/after 2026-03-01 ===")
display(after_cut)

# 4) If none, show the last 5 timestamps present for the EDA interval
last_ts = con.execute(
    "SELECT DISTINCT ts FROM ohlcv WHERE interval=? ORDER BY ts DESC LIMIT 5",
    [ITV_USED],
).df()
last_ts['ts'] = pd.to_datetime(last_ts['ts'])
print(f"\nLast 5 distinct timestamps for interval={ITV_USED}:")
display(last_ts)

# 5) Quick explanation hint
max_ts = max_by_symbol['max_ts'].max() if len(max_by_symbol) else None
if max_ts is not None:
    if max_ts < cut:
        print(
            f"\nInterpretation: the database itself only contains data through {max_ts}. "
            "So the EDA cannot show anything after March 1 because it is not in DuckDB.\n"
            "Most likely causes: loader was run with a fixed end date / limited months, "
            "or the source (Binance Vision monthly dumps) was only available through February for the period you loaded."
        )
    else:
        print(
            f"\nInterpretation: data exists after March 1 (max_ts={max_ts}); "
            "if your EDA stops earlier, it’s likely a filtering/imputation step dropping rows."
        )


## March 1 cutoff (reminder)

If the **wide** return matrix looked short, the returns cell uses forward-fill + `min_frac` so the common window matches most of the DB. Use the **diagnostic cell** for `MAX(ts)` by interval/symbol.

**Refresh data:** `research/db/README.md`


In [ ]:
#